# Pixel3DMM: одна кнопка → результат на Google Drive

Восстановление 3D-лица (FLAME) по фото: **предобработка → нормали → UV → трекинг → отдельная папка
на каждое фото**. Код библиотеки: [SimonGiebenhain/pixel3dmm](https://github.com/SimonGiebenhain/pixel3dmm).

## Три шага

| Шаг | Ячейка | Что происходит | Сколько ждать |
|---|---|---|---|
| 1 | **⚙️ Подготовка** | Drive → скрипты → окружение (Python 3.9 + CUDA-сборки) → веса → проверка → кеш | первый раз 25–40 мин, дальше 2–5 мин |
| 2 | **📤 Загрузить фото или архив** | кнопка выбора файла → план → реконструкция каждого фото → сохранение на Drive | ~1,5–2 мин на фото (T4) |
| 2б | *(необязательно)* | `REUSE_LAST = True` — повторить прошлую загрузку без нового окна: пересчитаются только неготовые фото | — |
| 3 | **📊 Статус и результаты** | что уже готово на Drive, превью, состав папки фото | мгновенно, без GPU |

**Что должно лежать на Drive заранее:** `FLAME2020.zip` и `FLAME2023.zip` — **в корне** `MyDrive`
(не в подпапке). Результаты: `MyDrive/pixel3dmm_results/<имя архива>/photos/<имя фото>/`.

## Как читать панель статуса

У каждого шага одна живая панель: этап и фаза, прогресс-бар с процентами и оценкой остатка,
**«сколько секунд назад был вывод»**, хвост лога активного трекера и кнопка **⏹ Остановить**.

- «вывода нет 1–3 мин» + хвост трекера меняется → идёт тяжёлый шаг, всё нормально;
- «вывода нет 10 мин» и лог трекера не растёт → жми «Остановить», результат уже посчитанных
  фото на Drive, повторный запуск продолжит с неготовых;
- остановка и обрыв сессии безопасны: каждое готовое фото лежит отдельной папкой с `done.json`,
  при повторном запуске такие фото пропускаются.

> Порядок не обязателен: если окружения нет, **ячейка 2 поставит его сама** (то же, что ячейка 1),
> и только потом спросит файлы. Ячейка 1 нужна, если хочется подготовиться заранее или проверить
> окружение без загрузки фото. Ячейки 2 и 3 можно запускать сколько угодно раз.

In [ ]:
# ⚙️ 1. ПОДГОТОВКА: Drive → скрипты → окружение → веса → проверка → кеш.
# Запускай один раз за сессию (после сброса рантайма — снова). Первый раз 25–40 минут,
# дальше 2–5 минут: окружение поднимается из кеша на Google Drive.
OUTPUT_ROOT    = "MyDrive/pixel3dmm_results"   # ← куда складывать результаты
SAVE_ENV_CACHE = True                          # кеш окружения на Drive (~10 ГБ) для быстрых сессий

import importlib, pathlib, sys, time, urllib.request
sys.path.insert(0, "/content")
ui_path = pathlib.Path("/content/p3dmm_ui.py")
tmp = ui_path.with_suffix(".py.new")
try:                                          # ВСЕГДА берём свежий модуль: правки подхватываются сами
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/hudyakovictor/colab/main/p3dmm_ui.py"
        f"?_={int(time.time())}", str(tmp))
    if tmp.stat().st_size > 20000:
        tmp.replace(ui_path)
    else:
        tmp.unlink(missing_ok=True)
except Exception as error:                    # нет сети/репозитория — работаем локальной копией
    print("свежий p3dmm_ui.py не скачался:", type(error).__name__, "— использую локальную копию")
    tmp.unlink(missing_ok=True)
    if not ui_path.is_file():
        import shutil
        from google.colab import drive
        drive.mount("/content/drive")
        try:
            shutil.copy2("/content/drive/MyDrive/p3dmm_ui.py", str(ui_path))
        except Exception:
            raise SystemExit("Нет p3dmm_ui.py ни в репозитории, ни в корне MyDrive. "
                             "Залей его в репозиторий hudyakovictor/colab (ветка main) или в MyDrive.")
import p3dmm_ui as ui
importlib.reload(ui)

with ui.Board("1 · Подготовка окружения Pixel3DMM") as board:
    result = ui.prepare(board, OUTPUT_ROOT, auto_cache=SAVE_ENV_CACHE)
    if result["problems"]:
        board.set(stats="замечания проверки: " + "; ".join(result["problems"])[:400])
    else:
        board.set(phase="Готово к работе",
                  stats=f"свободно {result.get('free_gb')} ГБ · FLAME2023: "
                        f"{'да' if result.get('flame2023') else 'нет'} · дальше — ячейка 2")


In [ ]:
# 📤 2. ЗАГРУЗИТЬ ФОТО ИЛИ АРХИВ → результаты на Google Drive.
# Нажми ▶︎ и выбери в окне браузера: одно фото, несколько фото или .zip с папкой фото.
OUTPUT_ROOT = "MyDrive/pixel3dmm_results"   # ← тот же путь, что в ячейке 1
MAX_PHOTOS  = 0                             # 0 = все фото архива; 1–3 — для пробного прогона
REUSE_LAST  = False                         # True = повторить прошлую загрузку без окна выбора
                                             # (досчитать неудачные фото, остальные пропустятся)
AUTO_PREPARE = True                         # окружения нет? ячейка 2 сама поставит его (как ячейка 1)

import importlib, pathlib, sys, time, urllib.request
sys.path.insert(0, "/content")
ui_path = pathlib.Path("/content/p3dmm_ui.py")
tmp = ui_path.with_suffix(".py.new")
try:                                          # ВСЕГДА берём свежий модуль: правки подхватываются сами
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/hudyakovictor/colab/main/p3dmm_ui.py"
        f"?_={int(time.time())}", str(tmp))
    if tmp.stat().st_size > 20000:
        tmp.replace(ui_path)
    else:
        tmp.unlink(missing_ok=True)
except Exception as error:                    # нет сети/репозитория — работаем локальной копией
    print("свежий p3dmm_ui.py не скачался:", type(error).__name__, "— использую локальную копию")
    tmp.unlink(missing_ok=True)
    if not ui_path.is_file():
        import shutil
        from google.colab import drive
        drive.mount("/content/drive")
        try:
            shutil.copy2("/content/drive/MyDrive/p3dmm_ui.py", str(ui_path))
        except Exception:
            raise SystemExit("Нет p3dmm_ui.py ни в репозитории, ни в корне MyDrive. "
                             "Залей его в репозиторий hudyakovictor/colab (ветка main) или в MyDrive.")
import p3dmm_ui as ui
importlib.reload(ui)

if not pathlib.Path("/content/p3dmm-env/bin/python").is_file():
    if not AUTO_PREPARE:
        raise RuntimeError("Окружения нет: сначала выполни ячейку 1 (после сброса рантайма /content пустой).")
    print("Окружения нет — ставлю его здесь же (та же логика, что в ячейке 1).", flush=True)
    with ui.Board("1 · Подготовка окружения (запущена из ячейки 2)") as prep:
        ui.prepare(prep, OUTPUT_ROOT, auto_cache=True)

with ui.Board("2 · Фото → FLAME-реконструкция → Google Drive") as board:
    summary = ui.process(board, OUTPUT_ROOT, max_photos=MAX_PHOTOS, reuse_last=REUSE_LAST)

from IPython.display import HTML, display
display(HTML(ui.status_html(summary)))


In [ ]:
# 📊 3. СТАТУС И РЕЗУЛЬТАТЫ — можно запускать в любой момент, даже без окружения и GPU:
# читает только Google Drive. Таблица готовых фото, превью последнего, состав папки фото.
OUTPUT_ROOT = "MyDrive/pixel3dmm_results"   # ← тот же путь, что в ячейках 1 и 2

import importlib, pathlib, sys, time, urllib.request
sys.path.insert(0, "/content")
ui_path = pathlib.Path("/content/p3dmm_ui.py")
tmp = ui_path.with_suffix(".py.new")
try:                                          # ВСЕГДА берём свежий модуль: правки подхватываются сами
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/hudyakovictor/colab/main/p3dmm_ui.py"
        f"?_={int(time.time())}", str(tmp))
    if tmp.stat().st_size > 20000:
        tmp.replace(ui_path)
    else:
        tmp.unlink(missing_ok=True)
except Exception as error:                    # нет сети/репозитория — работаем локальной копией
    print("свежий p3dmm_ui.py не скачался:", type(error).__name__, "— использую локальную копию")
    tmp.unlink(missing_ok=True)
    if not ui_path.is_file():
        import shutil
        from google.colab import drive
        drive.mount("/content/drive")
        try:
            shutil.copy2("/content/drive/MyDrive/p3dmm_ui.py", str(ui_path))
        except Exception:
            raise SystemExit("Нет p3dmm_ui.py ни в репозитории, ни в корне MyDrive. "
                             "Залей его в репозиторий hudyakovictor/colab (ветка main) или в MyDrive.")
import p3dmm_ui as ui
importlib.reload(ui)

from IPython.display import HTML, display

ui.mount_drive()
summary = ui.status(OUTPUT_ROOT)
display(HTML(ui.status_html(summary)))
ui.preview_last(OUTPUT_ROOT, count=1)
display(HTML(ui.legend_html()))

problems = ui.preflight_report(OUTPUT_ROOT).get("problems") or []
if problems:
    display(HTML('<section style="font:13px system-ui;border-left:5px solid #c0392b;padding:8px 12px;'
                 'border-radius:8px;background:#fdf6f6"><b>Проверка готовности нашла замечания:</b><br>'
                 + "<br>".join(problems) + '</section>'))


### Что извлекается в папку каждого фото

`MyDrive/pixel3dmm_results/<архив>/photos/<имя фото>/` — всё, что даёт библиотека по этому снимку:

- **`mesh.ply`** — 3D-меш лица в позе снимка (FLAME, 5023 вершины).
- **`canonical.ply`** — та же голова в канонической позе: без поворота и без мимики.
- **`params.json`** — все параметры: `flame` (`shape`, `exp`, `jaw`, `neck`, `eyes`, `eyelids`, `R`, `t`),
  `camera` (фокус, сдвиг), `pose` (yaw/pitch/roll + метка ракурса), `visibility` (что видно на снимке),
  конфиг трекинга и список файлов. Ракурс считается по матрице поворота, а не «на глаз».
- **`checkpoint.frame`** — сырой чекпоинт трекера (те же параметры без округления).
- **ландмарки:** `landmarks_68.npy`, `landmarks_7.npy`, `landmarks_eyes.npy` (результат трекинга),
  `landmarks_pipnet.npy` (98 точек PIPNet — детекция до трекинга).
- **идентичность:** `mica_identity.npy` (300 чисел MICA), `mica_mesh.ply`, `mica_kpt7.npy`, `mica_kpt68.npy`,
  `arcface.npy` (вектор лица для распознавания).
- **карты и маски:** `crop.jpg`, `segmentation.png`, `normals.png`, `uv_map.png`,
  `mask_fg.png`, `mask_uv.png`, `mask_normal.png`, `visibility_points.npy`.
- **превью:** `overlay.jpg` (кадр из `result.mp4` — меш поверх лица), `source.jpg` (исходник).
- **служебное:** `run.log` (итерации и лоссы трекинга), `done.json` (метка готовности).

Сводка по всем фото: `pixel3dmm_results/_index/photos.csv` (фото → статус, время, ракурс, нормы
параметров) и `_index/report.json`. Журнал попыток: `_state/photos_log.jsonl`.
Полный список с расшифровкой печатает ячейка 3 (`ui.legend_html()`).

### Частые ситуации

- **«Окружения нет: выполни ячейку 1»** — рантайм сбросился или сменился тип GPU, `/content` очистился.
  Ячейка 1 поднимет окружение из кеша на Drive за минуты.
- **Google Drive не подключился** — запусти ячейку 1 ещё раз и разреши доступ в окне браузера.
- **Нет `FLAME2020.zip` / `FLAME2023.zip`** — оба архива должны лежать в корне `MyDrive`
  (путь `MyDrive/FLAME2020.zip`), не в подпапке.
- **Фото без лица в кадре** — отсеиваются на этапе плана (`face_score < 0.75`), папки результата у них
  не будет; список отброшенных печатается в лог плана.
- **В имени файла нет даты** — не проблема: дата берётся из EXIF, затем из времени файла, поэтому
  хронологический порядок сохраняется, а фото не выбрасываются.
- **Повторить конкретное фото** — удали на Drive его папку `photos/<имя фото>/`, поставь в ячейке 2
  `REUSE_LAST = True` и запусти: остальные пропустятся, пересчитается только удалённое
  (или без удаления — досчитаются только неготовые после таймаутов).
- **Сессия оборвалась по лимиту Colab** — это нормально: каждое готовое фото уже на Drive.
  Снова ячейка 1 → 2, обработка продолжится с неготовых.
- **Место на Drive** — кеш окружения занимает ~10 ГБ; если он не нужен, поставь
  `SAVE_ENV_CACHE = False` в ячейке 1.